# 02 · Bronze → Silver

Conceitos praticados:
- **Deduplicação** (`dropDuplicates` por chave de negócio)
- **Tratamento de nulos** (nulos estruturais vs. nulos de negócio — nem todo nulo deve ser preenchido!)
- **Conversão de formatos** (string → timestamp, string → decimal/double)
- **Validações de qualidade de dados** com um log de resultados persistido em Delta

In [0]:
catalog = "cine_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime

dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def converter_data(data_str, formatos_permitidos=None):
    if not formatos_permitidos:
        # Lista dos formatos mais comuns encontrados em bases de dados
        formatos_permitidos = [
            "%d-%m-%Y",
            "%Y-%m-%d",
            "%d/%m/%Y",
            "%Y/%m/%d",
            "%m/%d/%Y",
            "%m-%d-%Y",
        ]

    data_str = data_str.strip()
    for formato in formatos_permitidos:
        try:
            return datetime.strptime(data_str, formato)
        except ValueError:
            pass
    return None

## 1. info

In [0]:
from pyspark.sql.window import Window

df_bronze_info = spark.table(f"{bronze_schema}.info")

_w = Window.partitionBy("id").orderBy(F.col("ingestion_timestamp").desc())

df_silver_info = (
    df_bronze_info
    .withColumn("_rn", F.row_number().over(_w))
    .filter((F.col("_rn") == 1) & F.col("id").isNotNull())
    .drop("_rn")
    .withColumn("original_language", F.upper(F.trim(F.col("original_language"))))
    .withColumn("status", F.lower(F.trim(F.col("status"))))
    # .withColumn("release_date", converter_data(F.col("release_date")))

    .withColumn("status",
        F.when(F.col("status") == "released", "Lançado")
        .when(F.col("status") == "post production", "Pós-Produção")
        .when(F.col("status") == "in production", "Em Produção")
        .when(F.col("status") == "planned", "Planejado")
        .when(F.col("status") == "rumored", "Rumores")
        .when(F.col("status") == "canceled", "Cancelado")
        .otherwise("Não Informado")
    )

    .withColumn("release_date",
        F.date_format(
            F.coalesce(
                F.try_to_date(F.col("release_date"), "dd-MM-yyyy"),
                F.try_to_date(F.col("release_date"), "yyyy-MM-dd"),
                F.try_to_date(F.col("release_date"), "dd/MM/yyyy"),
                F.try_to_date(F.col("release_date"), "yyyy/MM/dd"),
                F.try_to_date(F.col("release_date"), "MM/dd/yyyy"),
                F.try_to_date(F.col("release_date"), "MM-dd-yyyy"),
            ),
            "dd-MM-yyyy"
        )
    )

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")

    .withColumn("ano_lancamento", F.year(F.try_to_date(F.col("data_lancamento"), "dd-MM-yyyy")))
)

dq_check("silver.info", "id_filme não nulo", df_silver_info, F.col("id_filme").isNotNull())
dq_check_unique("silver.info", "id_filme único", df_silver_info, ["id_filme"])

df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.info")

display(df_bronze_info.limit(10))
display(df_silver_info.limit(10))

[PASS] silver.info | id_filme não nulo | 0/97879 linhas falharam
[PASS] silver.info | id_filme único | 0 chaves duplicadas de 97879 linhas


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_timestamp
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-16T11:12:05.661Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-16T11:12:05.661Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-16T11:12:05.661Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-16T11:12:05.661Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-16T11:12:05.661Z
284054,tt1825683,Black Panther,Black Panther,en,2018-02-13,135,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null,2026-10-16T11:12:05.661Z
284052,tt1211837,Doctor Strange,Doctor Strange,en,2016-10-25,115,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.,2026-10-16T11:12:05.661Z
315635,tt2250912,Spider-Man: Homecoming,Spider-Man: Homecoming,en,2017-07-05,133,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.,2026-10-16T11:12:05.661Z
283995,tt3896198,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,en,2017-04-19,137,Released,The Guardians must fight to keep their newfound family together as 

id_filme,tconst,titulo,titulo_original,idioma_original,data_lancamento,duracao_minutos,status_filme,sinopse,frase_divulgacao,ingestion_timestamp,ano_lancamento
14564,tt0498381,Rings,Rings,EN,01-02-2017,102,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die.",null,2026-10-16T11:12:05.661Z,2017
32471,tt1587420,Mixtape,Mixtape,EN,03-12-2021,94,Lançado,null,null,2026-10-16T11:12:05.661Z,2021
38258,tt0093119,Grizzly II: Revenge,Grizzly II: Revenge,EN,17-02-2020,74,Lançado,\All hell breaks loose when a giant grizzly,reacting to the slaughter of her cubs by poachers,2026-10-16T11:12:05.661Z,2020
38492,tt0215569,Billy Joel - Live at Yankee Stadium,Billy Joel Live at Yankee Stadium,EN,22-06-2022,86,Lançado,Billy Joel plays his greatest hits in the Big Apple.,null,2026-10-16T11:12:05.661Z,2022
38700,tt1502397,Bad Boys for Life,Bad Boys for Life,EN,15-01-2020,124,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel.",Ride together. Die together.,2026-10-16T11:12:05.661Z,2020
42018,tt0090907,The Horse Thief,盗马贼,ZH,19-03-2019,88,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter.",null,2026-10-16T11:12:05.661Z,2019
42330,tt10443316,Monkey Magic,大闹西游,ZH,22-09-2018,66,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3",null,2026-10-16T11:12:05.661Z,2018
43074,tt1289401,Ghostbusters,Ghostbusters,EN,14-07-2016,117,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat.",Who You Gonna Call?,2026-10-16T11:12:05.661Z,2016
45033,tt1202339,20 Seconds of Joy,20 Seconds of Joy,DE,01-01-2018,60,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear.",null,2026-10-16T11:12:05.661Z,2018
46983,tt0051471,The Song of Styrene,Le Chant du styrène,FR,23-05-2022,13,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.,null,2026-10-16T11:12:05.661Z,2022


## 2. financials

## 3. metrics

## 4. reviews

## 5. genres

## 6. companies

## 7. dolar